### config

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.connect.functions import to_date, unix_timestamp, udf, lit
from pyspark.sql.functions import column
import openmeteo_requests
from retry_requests import retry
import requests_cache
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, MapType, DoubleType
import math

import config.ConnectionConfig as cc


cc.setupEnvironment()

spark = cc.startLocalCluster("fact_rides",7)
spark.getActiveSession()

25/05/09 14:01:22 WARN Utils: Your hostname, 4L3KS-comp resolves to a loopback address: 127.0.1.1; using 10.140.99.119 instead (on interface wlp2s0)
25/05/09 14:01:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/aleks/Downloads/bigtools/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/aleks/.ivy2/cache
The jars for the packages stored in: /home/aleks/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-57c77ce3-c4a2-4c0f-b167-8f4a85d3a3d2;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.

# EXTRACT

I tried using the spark API first, but there are some limitations on their joining of columns.

In [5]:
#Weather data stuff
def fetch_weather_data(end_date:str, latitude: float, longitude: float):
    """
    Fetch weather data for a specific hour at the given location.

    :param end_date: End date, function will save information about last 30 days from this date
    :param latitude: Latitude of the location.
    :param longitude: Longitude of the location.
    :return Returns a dataframe with temperature and weather data or an error message.
    """
    # Setup API client with caching and retry mechanism
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    # Calculate start date
    end_date_dt = datetime.strptime(end_date, "%Y-%m-%d")
    start_date_dt = end_date_dt - timedelta(days=30)
    start_date = start_date_dt.strftime("%Y-%m-%d")

    # Extract date portion for API request

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["temperature_2m", "weather_code"],
        "timezone": "auto"
    }

    responses = openmeteo.weather_api(url, params=params)
    if not responses:
        # Return an empty dataframe with the correct column names
        return pd.DataFrame({
            "date": [],
            "temperature_2m": [],
            "weather_code": [],
            "error": ["No data available for the specified dates"]
        })

    response = responses[0]

    hourly = response.Hourly()
    hourly_timestamps = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )

    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_weather_code = hourly.Variables(1).ValuesAsNumpy()

    df = pd.DataFrame({
        "date": hourly_timestamps,
        "temperature_2m": hourly_temperature_2m,
        "weather_code": hourly_weather_code
    })

    return df


def to_weather_code_convertion(weather_dict):
    temp = weather_dict["temperature_2m"]
    code = weather_dict["weather_code"]
    print(f"Temperature: {temp}. Code: {code}")
    if float(temp) > 14 and float(code) < 1:
        return 2 # Pleasant
    elif float(code) > 50:
        return 1 # Unpleasant
    else:
        return 3 # Neutral

# Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = math.sin(delta_phi / 2.0) ** 2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2.0) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

# Register the UDF
haversine_udf = udf(haversine, DoubleType())
spark.udf.register("haversine_km", haversine_udf)


def convert_to_lat_lon(coord_str):
    # Remove parentheses and split the string by comma
    coord_str = coord_str.strip("()")
    latitude, longitude = map(float, coord_str.split(","))
    return latitude, longitude

def convert_to_nearest_hour(starttime):
    starttime_rounded = starttime.replace(minute=0, second=0, microsecond=0)
    return starttime_rounded


seasonal_dates = ["2019-02-22", "2019-06-22", "2019-09-22"]  # Winter, Summer, Autumn  # Use real `starttime` from rides
last_date_col = to_date(lit(seasonal_dates[:10]))  # Extract "YYYY-MM-DD" part

# Registering a function
haversine_udf = udf(haversine, DoubleType())
spark.udf.register("haversine_km", haversine_udf)

weather_schema = StructType([
    StructField("rideid", IntegerType(), True),
    StructField("weather", IntegerType(), True)
])

/home/aleks/PycharmProjects/sparkdelta/.venv/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.4 is exactly one major version older than the runtime version 6.30.2 at google/protobuf/duration.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
25/05/09 14:04:40 WARN SimpleFunctionRegistry: The function haversine_km replaced a previously registered function.


In [2]:
# EXTRACT rides:
# rides_table_SQL = '(SELECT * FROM rides) as rides_table'
# df_rides = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", rides_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_rides.printSchema()
#df_rides.show()
#df_rides.count()
# ----

#EXTRACT bike_type through bike_lot through vehicle:
# Rename to avoid duplicate column names
# Corrected column name
# vehicle_table_SQL = """
# (
#     SELECT
#         v.*,
#         l.bikelotid AS bikelotid_bikelots,
#         l.deliverydate,
#         l.biketypeid
#     FROM vehicles v
#     LEFT JOIN bikelots l ON v.bikelotid = l.bikelotid
# ) AS vehicle_table
# """
#
# df_vehicles = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", vehicle_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "vehicleid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_vehicles.show()
# ------

#EXTRACT users through subscriptionid through subscription
# user_table_SQL = """
# (
#     SELECT
#         userid AS userid_subscriptions,
#         subscriptionid AS subscriptionid_subscriptions
#     FROM subscriptions s
# ) AS user_table
# """
#
# df_users = spark.read.format("jdbc")\
#     .option("driver", cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", user_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "userid_subscriptions") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()

#df_users.show()

# ------

#EXTRACT date
# df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
# df_dates.show()
# df_dates.createOrReplaceTempView("dates")

# ------

#EXTRACT weather: TBD.



Trying to do it in one SQL query:

In [6]:
# EXTRACTING ALL IN ONE GO:
# load dates from deltatable:
df_dates = spark.read.format("delta").load("../dimensions/spark-warehouse/dimdate")
#df_dates.show()
df_dates.createOrReplaceTempView("dates")

# write the SQL string needed:
SQL = """(
    SELECT
    r.*,
    v.vehicleid AS vehicleid_vehicles,
    v.bikelotid AS bikelotid_vehicles,
    b.bikelotid AS bikelotid_bikelots,
    b.biketypeid AS biketypeid_bikelots,
    s.subscriptionid AS subscriptionid_subscriptions,

    s.userid AS userid_subscriptions
        FROM rides r
            LEFT JOIN vehicles v ON r.vehicleid = v.vehicleid
            LEFT JOIN bikelots b ON v.bikelotid = b.bikelotid
            LEFT JOIN subscriptions s ON r.subscriptionid = s.subscriptionid

) as rides_table
"""

df_rides_full = spark.read.format("jdbc")\
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", SQL) \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0)\
    .option("upperBound", 100) \
    .load()

df_rides_full.show()
# WEATHER would be done separately


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|vehicleid_vehicles|bikelotid_vehicles|bikelotid_bikelots|biketypeid_bikelots|subscriptionid_subscriptions|userid_subscriptions|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+------------------+------------------+------------------+-------------------+----------------------------+--------------------+
|     1|(51.2083,4.44595)|(51.1938,4.40228)|2015-09-22 00:00:00|2012-09-22 00:00:00|      844|         13296|       4849|     3188|               844|                 3|                 3|                  1|               

In [14]:
#EXTRACT
df_users = spark.read.load('./spark-warehouse/dim_user/dim_user.snappy.parquet')
df_users.createOrReplaceTempView("users_dim")
df_dates = spark.read.load('./spark-warehouse/dimdate/dim_date.snappy.parquet')
df_dates.createOrReplaceTempView("dates_dim")
df_locks = spark.read.load('./spark-warehouse/dimlock_parquet/dimLock_parquet.snappy.parquet')
df_locks.createOrReplaceTempView("locks_dim")
df_locks.show()


+-------+----------+---------+--------------------+------+-------+---------+-----------------+
|lock_id|station_id|stationnr|              street|number|zipcode| district|        gps_coord|
+-------+----------+---------+--------------------+------+-------+---------+-----------------+
|   9999|      9999|     None|                None|  None|   None|     None|              0,0|
|    217|        12|      120|Schijnpoortweg (2...| 27-29|   2060|ANTWERPEN|(51.2276,4.43923)|
|    218|        12|      120|Schijnpoortweg (2...| 27-29|   2060|ANTWERPEN|(51.2276,4.43923)|
|    219|        12|      120|Schijnpoortweg (2...| 27-29|   2060|ANTWERPEN|(51.2276,4.43923)|
|    220|        12|      120|Schijnpoortweg (2...| 27-29|   2060|ANTWERPEN|(51.2276,4.43923)|
|    221|        12|      120|Schijnpoortweg (2...| 27-29|   2060|ANTWERPEN|(51.2276,4.43923)|
|    222|        12|      120|Schijnpoortweg (2...| 27-29|   2060|ANTWERPEN|(51.2276,4.43923)|
|    223|        12|      120|Schijnpoortweg (2...

# TRANSFORM

In [9]:
#TRANSFORMATION
date_joined_df = df_users.join(df_rides_full,
        (df_users.subscriptionid == df_rides_full.subscriptionid_subscriptions) &
        (df_users.validfrom  <= df_rides_full.starttime) &
        (df_users.end_subscription_date >= df_rides_full.starttime),"inner")

joined_df = (
    date_joined_df
    .join(df_dates, df_dates.CalendarDate == date_joined_df.starttime.cast('date'), "leftouter")
    .select(
        df_dates.dateSK.alias("dates_sk"),
        df_rides_full.rideid,
        df_rides_full.startlockid.alias("start_lock_id"),
        df_rides_full.endlockid.alias("end_lock_id"),
        df_rides_full.vehicleid.alias("vehicle_id"),
        df_users.user_sk.alias("user_sk"),
        (df_rides_full.endtime.cast("long") - df_rides_full.starttime.cast("long")).alias("duration") #this is in seconds, turn into minutes probs
    )
)
joined_df.show()


+--------+------+-------------+-----------+----------+-------+--------+
|dates_sk|rideid|start_lock_id|end_lock_id|vehicle_id|user_sk|duration|
+--------+------+-------------+-----------+----------+-------+--------+
|       2|    15|         4849|       3188|       844|   6655|     893|
|       2|    16|         NULL|       NULL|      4545|  23057|     124|
|       2|    18|         1821|       2186|      1208|  15450|     304|
|       2|    22|         2572|         13|      2015|  32680|     335|
|       2|    23|           50|       2067|      5298|  35667|     395|
|       2|    25|          985|       2148|      1400|    510|     271|
|       2|    26|         2039|       3038|       957|  30033|     641|
|       2|    27|         5619|       2717|      5413|   2561|    1176|
|       2|    28|         3531|       3554|      2658|  33017|       0|
|       2|    29|         4940|       1218|      1925|  19276|     194|
|       2|    32|         1056|        792|      3572|  14820|  

In [ ]:
#Weather api transform
import os
import json
os.makedirs("weather", exist_ok=True)

wmo_to_main = {
    0: ("Clear", "clear sky", "01d"),
    1: ("Mainly clear", "few clouds", "02d"),
    2: ("Partly cloudy", "scattered clouds", "03d"),
    3: ("Overcast", "overcast clouds", "04d"),
    45: ("Fog", "fog", "50d"),
    48: ("Rime fog", "rime fog", "50d"),
    51: ("Drizzle", "light drizzle", "09d"),
    53: ("Drizzle", "moderate drizzle", "09d"),
    55: ("Drizzle", "dense drizzle", "09d"),
    61: ("Rain", "slight rain", "10d"),
    63: ("Rain", "moderate rain", "10d"),
    65: ("Rain", "heavy rain", "10d"),
    71: ("Snow", "slight snow", "13d"),
    73: ("Snow", "moderate snow", "13d"),
    75: ("Snow", "heavy snow", "13d"),
    80: ("Rain showers", "slight rain showers", "09d"),
    81: ("Rain showers", "moderate rain showers", "09d"),
    82: ("Rain showers", "violent rain showers", "09d"),
    95: ("Thunderstorm", "thunderstorm", "11d")
}

weather_by_zipcode = {}

for date_str in seasonal_dates:
    for row in df_rides_full.collect():
        lat, lon = convert_to_lat_lon(row.gpscoord)
        weather_data = fetch_weather_data(date_str, lat, lon)
        if not weather_data.empty:
            weather_data["zipcode"] = row.zipcode

            for _, weather_row in weather_data.iterrows():
                timestamp = weather_row["date"]
                key = (row.zipcode, timestamp.strftime("%Y-%m-%d %H:%M:%S"))
                weather_by_zipcode[key] = weather_row

                code = int(weather_row["weather_code"])
                temp_raw = weather_row.get("temperature_2m", None)
                try:
                    temp = round(float(temp_raw), 1) if temp_raw is not None else None
                except (ValueError, TypeError):
                    temp = None

                main, description, icon = wmo_to_main.get(code, ("Unknown", "unknown", "50d"))

                json_obj = {
                    "zipCode": int(row.zipcode),
                    "coord": {"lon": lon, "lat": lat},
                    "weather": [{
                        "id": code,
                        "main": main,
                        "description": description,
                        "icon": icon
                    }],
                    "base": "stations",
                    "dt": timestamp.isoformat(),
                    "temp": temp
                }

                file_name = f"weather/{row.zipcode}_{timestamp.strftime('%Y-%m-%dT%H-%M')}.json"
                with open(file_name, "w") as f:
                    json.dump(json_obj, f, indent=4)

rides_filtered = df_rides_full.filter(
    to_date(df_rides_full["starttime"]).isin(seasonal_dates)
).select("rideid", "starttime", "startlockid")

rides_with_locks = rides_filtered.join(
    df_locks,
    rides_filtered.startlockid == df_locks.lockid,
    "left"
).withColumnRenamed("stationid", "lock_stationid")

# LOAD

In [13]:
joined_df.repartition(1).write.format("parquet").mode("overwrite").saveAsTable('temp_fact_parquet')

In [14]:
spark.stop()